In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / 'dataset').exists() and (candidate / 'paths.py').exists():
        candidate_str = str(candidate.resolve())
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break


## Pancreatic Cancer Patient Info Extraction

### Imports

In [ ]:
from typing import Union, Literal, List
from pydantic import BaseModel, Field
from openai import OpenAI
import json
import warnings
from tqdm import tqdm

from dataset.mimic_dataset import MIMIC_Dataset
from evaluations.preprocess import PatientGroundTruth
from termcolor import colored
from paths import PANCREATIC_CANCER_INFO_PATH

from dotenv import load_dotenv
load_dotenv()

warnings.filterwarnings("ignore")

In [ ]:
OUTPUT_FILE_NAME = "pancreatic_cancer_info" # name the file to be generated (ideally do not change)
OUTPUT_FILE_PATH = PANCREATIC_CANCER_INFO_PATH.with_name(f"{OUTPUT_FILE_NAME}.json")

### Information Extraction Classes

In [ ]:
class ERCP(BaseModel):
    """Information about the ERCP procedure."""

    has_ercp: bool = Field(
        ...,
        description="Whether the patient has had an ERCP during the **current** hospitalization.",
    )
    biopsy_result: Union[Literal[False], str] = Field(
        ...,
        description="The result of the biopsy. False if no biopsy was performed in the **current** hospitalization, otherwise the result of the biopsy as a string in the exact same words as stated in the patient information.",
    )


class Imaging(BaseModel):
    """Information about the imaging."""

    imaging_type: Literal[
        "Radiograph",
        "CT",
        "Ultrasound",
        "MRI",
        "Mammogram",
        "CTU",
        "Fluoroscopy",
        "Carotid ultrasound",
        "Paracentesis",
        "MRCP",
        "Upper GI Series",
        "Drainage",
        "MRE",
        "MRA",
        "ERCP",
        "PTC",
    ] = Field(
        ...,
        description="The type of imaging.",
    )
    region: Literal[
        "Chest",
        "Abdomen",
        "Head",
        "Spine",
        "Venous",
        "Knee",
        "Neck",
        "Foot",
        "Shoulder",
        "Ankle",
        "Wrist",
        "Hand",
        "Hip",
        "Finger",
        "Femur",
        "Bone",
        "Scrotum",
        "Heel",
        "Thigh",
    ] = Field(
        ...,
        description="The region of the body that was imaged in the study.",
    )
    result: str = Field(
        ...,
        description="The result of the imaging as a string in the exact same words as stated in the patient information.",
    )


class Diagnosis(BaseModel):
    """Information about the diagnosis of pancreatic cancer."""

    diagnosis_status: bool = Field(
        ...,
        description="Whether the patient has already a **secured** diagnosis of pancreatic cancer BEFORE being admitted for the **current** hospitalization. True if yes, False if the patient comes with unclear abdominal symptoms or suspicion of pancreatic cancer to complete the diagnostic workup during their hospitalization.",
    )
    external_staging: Union[Literal[False], List[Imaging]] = Field(
        ...,
        description="Completely ignore any imaging or staging done during the **current** hospital stay. False if no **external** (CT or MRI) imaging was mentioned, otherwise a list of all the types and results of the imaging(s) in the exact same words as stated in the patient information. Imaging mentioned during the hospital stay must not be included in your evaluation. If no previous imaging is mentioned, but imaging during the hospital stay was done, return False.",
    )


class Information(BaseModel):
    """Extracted information about the patient."""

    has_diagnosis: Diagnosis = Field(
        ...,
        description="Whether the patient has already a completed diagnosis of pancreatic cancer when admitted to the hospital. True if yes, False if the patient comes to complete the diagnosis during their hospitalization. If True, also check if external imaging results are provided and quote them.",
    )
    admission_reason: str = Field(
        ...,
        description="The reason for the patient's admission as a string in clear sentences (shortly, but must contain all relevant information). For instance, if the patient came for a planned Whipple surgery, state this. If the patient came because of unclear symptoms (or suspected pancreatic cancer), state that it was for completion of the diagnosis.",
    )
    existing_info: str = Field(
        ...,
        description="All staging results that already exist (imaging, ERCP, laparoscopy, etc) **before** the current admission including biopsy results. Focus on `History of Present Illness` and `Past Medical History` sections. You must not include any new information that was obtained during the current hospital admission.",
    )
    has_whipple: bool = Field(
        ...,
        description="Whether the patient has had a Whipple procedure during their hospitalization.",
    )
    ercp: ERCP = Field(
        ...,
        description="Whether the patient has had an ERCP during their **current** hospitalization and the result of the biopsy (if any).",
    )

    class Config:
        arbitrary_types_allowed = True

### Run the Extraction

In [ ]:
client = OpenAI()
ds = MIMIC_Dataset.load_dataset("pancreatic_cancer")

def extract_information(patient_info: str):
    completion = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Extract the relevant information."},
            {
                "role": "user",
                "content": f"Patient information:\n\n{patient_info}",
            },
        ],
        response_format=Information,
    )

    return completion.choices[0].message.parsed

In [ ]:
patient_infos = {}
for idx, pat in enumerate(tqdm(ds)):
    patient_info = pat.history_pe_admedication_diagnosis["text"].values[0].strip()
    info = extract_information(patient_info)
    patient_gt = PatientGroundTruth(pat)
    radiology_events = patient_gt.radiology_events

    if isinstance(info.has_diagnosis.external_staging, list):
        new_stagings = [
            imaging
            for imaging in info.has_diagnosis.external_staging
            if not any(
                imaging.imaging_type == event["modality"]
                and imaging.region == event["region"]
                for event in radiology_events
            )
            and not isinstance(imaging, bool)
        ]
        info.has_diagnosis.external_staging = new_stagings
    patient_infos[pat.hadm_id] = info.model_dump()

In [ ]:
# save the results
OUTPUT_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_FILE_PATH, "w") as f:
    json.dump(patient_infos, f)
print(f"Saved: {OUTPUT_FILE_PATH}")

### Show one example

In [ ]:
print(f"Diagnosis status: {colored(info.has_diagnosis.diagnosis_status, 'green')}")
print(f"Admission reason: {colored(info.admission_reason, 'green')}")
print(f"Existing info: {colored(info.existing_info, 'green')}")

print("External imaging:")
for imaging in info.has_diagnosis.external_staging:
    print(
        f"{colored(imaging.imaging_type, 'green')} - {colored(imaging.region, 'green')}: {colored(imaging.result, 'green')}"
    )

print(f"Whipple: {colored(info.has_whipple, 'green')}")
print(f"ERCP: {colored(info.ercp.has_ercp, 'green')}")
print(f"ERCP biopsy result: {colored(info.ercp.biopsy_result, 'green')}")